### RAG Retrieval Example
### This notebook demonstrates how to:
#### 1. Read a PDF resume
#### 2. Split it into chunks
#### 3. Store it in Chroma vector DB
#### 4. Retrieve relevant chunks for a query


In [2]:
!pip install chromadb pypdf sentence-transformers --quiet


##### 1) Import libraries

In [5]:
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions


##### 2) Function to read PDF

In [6]:
def read_pdf(path: str) -> str:
    """
    Read PDF file and return its full text
    """
    reader = PdfReader(path)
    text = ""
    
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    
    return text

##### 3) Function to split text into chunks

In [7]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50):
    """
    Split text into smaller chunks for better retrieval
    chunk_size = number of words per chunk
    overlap = number of overlapping words between chunks
    """
    words = text.split()
    chunks = []
    
    if not words:
        return chunks
    
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

##### 4) Function to create Chroma collection

In [8]:
def build_collection():
    """
    Create a Chroma collection with embeddings using SentenceTransformer
    """
    client = chromadb.Client()
    
    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    
    collection = client.create_collection(
        name="resume_collection",
        embedding_function=embedding_func
    )
    
    return collection

##### 5) Function to index resume into the vector DB

In [9]:
def index_resume(collection, resume_path: str, resume_id: str = "resume_1"):
    """
    Read PDF, chunk it, and add to Chroma collection
    """
    full_text = read_pdf(resume_path)
    chunks = chunk_text(full_text, chunk_size=200, overlap=50)
    
    ids = [f"{resume_id}_chunk_{i}" for i in range(len(chunks))]
    
    collection.add(
        documents=chunks,
        ids=ids
    )
    
    print(f"Indexed {len(chunks)} chunks from resume: {resume_path}")

##### 6) Function to retrieve relevant chunks

In [10]:
def retrieve(collection, query: str, top_k: int = 3):
    """
    Retrieve top_k most relevant chunks for a query
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    
    docs = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]
    ids = results.get("ids", [[]])[0]
    
    return list(zip(ids, docs, distances))

##### 7) Demo usage

In [ ]:
# Resume file path
resume_file = "resume.pdf"  

# 1) Build collection
collection = build_collection()

# 2) Index the resume
index_resume(collection, resume_file, resume_id="candidate_1")

# 3) Try retrieval
query = "what are the strengths and skills of this candidate?"
results = retrieve(collection, query, top_k=3)

# Print results
print("\nTop retrieved chunks:\n")
for i, (doc_id, doc_text, dist) in enumerate(results, start=1):
    print(f"Result {i}:")
    print(f"ID: {doc_id}")
    print(f"Distance: {dist:.4f}")
    print(f"Text snippet: {doc_text[:300]}...")
    print("-" * 80)


Indexed 2 chunks from resume: resume.pdf

Top retrieved chunks:

Result 1:
ID: candidate_1_chunk_0
Distance: 0.6843
Text snippet: MOHAMMAD SAYEH OBJECTIVE Passionate and hardworking Computer Science student with a strong foundation in Data Structures, Algorithms, and Object-Oriented Programming. Eager to apply technical knowledge and problem-solving skills in real-world projects while continuously developing new abilities. EDU...
--------------------------------------------------------------------------------
Result 2:
ID: candidate_1_chunk_1
Distance: 0.7958
Text snippet: inserting, updating, deleting, and searching both location and martyr records. Implemented a statistics module to generate reports on martyr counts, date analysis, and full data traversal. Supported saving updated information back to a CSV file with user-selected paths. Big-Data-Tweets-Project Respo...
--------------------------------------------------------------------------------
